# Assignment 2 — inspect every function

Private study aid, not a required submission file. Select the project's `.venv` kernel, restart it and run cells from top to bottom. This notebook calls the real functions in `genereport.py`; it does not replace them with mock responses or install packages. Live cells need internet access. On a request failure, stop and retry that cell later rather than treating missing network data as a missing biological annotation.

Each function has its own labelled cell, including the CLI entry point. Raw responses are deliberately shown before extracted values. Open the printed record links and compare the same fields, organism, accession/version and assembly. Database annotations can change: the earlier APOE example had Gene ID 348, five RefSeq mRNAs, UniProt P02649 and 39 Ensembl transcripts, but counts and annotations must be checked against today's responses rather than assumed constant.

All writing uses temporary directories. The saved report is read only; the last cell verifies it and the script stayed unchanged. Some outputs are long so you can inspect complete annotations instead of truncated summaries.


In [2]:
import difflib
import hashlib
import json
import re
import subprocess
import sys
import tempfile
import time
from datetime import datetime, timezone
from pathlib import Path
from pprint import pprint

assignment_dir = Path.cwd()
if not (assignment_dir / 'genereport.py').is_file():
    assignment_dir = assignment_dir / 'assignment_2'
assert (assignment_dir / 'genereport.py').is_file(), 'Open from the project root or assignment_2.'
sys.path.insert(0, str(assignment_dir))
import requests
import genereport as gr

symbol = 'APOE'
organism = 'Homo sapiens'
species = '_'.join(organism.lower().split())
saved_report_path = assignment_dir / 'reports' / 'APOE_Homo_sapiens.md'
protected_hashes = {}
for path in (assignment_dir / 'genereport.py', saved_report_path):
    if path.is_file():
        protected_hashes[path] = hashlib.sha256(path.read_bytes()).hexdigest()
print('Python:', sys.executable)
print('Script:', gr.__file__)
print('Session started (UTC):', datetime.now(timezone.utc).isoformat())


Python: d:\Studiowe\DataScience\Sem 1\Bioinformatics\Projekt 1\.venv\Scripts\python.exe
Script: d:\Studiowe\DataScience\Sem 1\Bioinformatics\Projekt 1\assignment_2\genereport.py
Session started (UTC): 2026-11-10T10:26:08.190619+00:00


## 1. `get_ncbi_json()` — raw NCBI search

Output: the complete search JSON. Inspect `esearchresult.count`, `idlist` and the translated query. Gene ID 348 is the previously verified human APOE example. The printed URL opens the same search in your browser.


In [3]:
search_parameters = {
    'db': 'gene',
    'term': f'"{symbol}"[Gene Name] AND "{organism}"[Organism]',
    'retmode': 'json', 'retmax': 2,
}
search_json = gr.get_ncbi_json('esearch.fcgi', search_parameters)
print(requests.Request('GET', gr.EUTILS_URL + 'esearch.fcgi', params=search_parameters).prepare().url)
pprint(search_json, sort_dicts=False)


https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=gene&term=%22APOE%22%5BGene+Name%5D+AND+%22Homo+sapiens%22%5BOrganism%5D&retmode=json&retmax=2
{'header': {'type': 'esearch', 'version': '0.3'},
 'esearchresult': {'count': '1',
                   'retmax': '1',
                   'retstart': '0',
                   'idlist': ['348'],
                   'translationset': [],
                   'translationstack': [{'term': '"APOE"[Gene Name]',
                                         'field': 'Gene Name',
                                         'count': '412',
                                         'explode': 'N'},
                                        {'term': '"Homo sapiens"[Organism]',
                                         'field': 'Organism',
                                         'count': '360697',
                                         'explode': 'Y'},
                                        'AND'],
                   'querytranslation': '"APOE"[Gene Name] 

## 2. `find_gene_id()` — the selected identifier

Compare the function's output with the raw ID list above. The script rejects zero or ambiguous matches; it should not silently choose the first of several genes.


In [4]:
gene_id = gr.find_gene_id(symbol, organism)
print('Function output:', gene_id)
print('Raw search IDs:', search_json['esearchresult']['idlist'])
assert search_json['esearchresult']['idlist'] == [gene_id]
print('Online gene record:', 'https://www.ncbi.nlm.nih.gov/gene/' + gene_id)


Function output: 348
Raw search IDs: ['348']
Online gene record: https://www.ncbi.nlm.nih.gov/gene/348


## 3. `fetch_gene_summary()` — validated gene record

The first output is the raw ESummary wrapper; the second is the function's returned record. Compare official symbol, scientific organism name and Gene ID. Expand `summary`, `mim`, chromosome and map location for the next cell.


In [5]:
summary_json = gr.get_ncbi_json('esummary.fcgi', {'db': 'gene', 'id': gene_id, 'retmode': 'json'})
print('RAW ESUMMARY:')
pprint(summary_json, sort_dicts=False)
gene_summary = gr.fetch_gene_summary(gene_id, symbol, organism)
print('\nFUNCTION OUTPUT:')
pprint(gene_summary, sort_dicts=False)
assert gene_summary == summary_json['result'][gene_id]


RAW ESUMMARY:
{'header': {'type': 'esummary', 'version': '0.3'},
 'result': {'uids': ['348'],
            '348': {'uid': '348',
                    'name': 'APOE',
                    'description': 'apolipoprotein E',
                    'status': '',
                    'currentid': '',
                    'chromosome': '19',
                    'geneticsource': 'genomic',
                    'maplocation': '19q13.32',
                    'otheraliases': 'AD2, APO-E, ApoE4, LDLCQ5, LPG',
                    'otherdesignations': 'apolipoprotein E|apolipoprotein E3',
                    'nomenclaturesymbol': 'APOE',
                    'nomenclaturename': 'apolipoprotein E',
                    'nomenclaturestatus': 'Official',
                    'mim': ['107741'],
                    'genomicinfo': [{'chrloc': '19',
                                     'chraccver': 'NC_000019.10',
                                     'chrstart': 44905795,
                                     'chrstop

## 4. `get_ncbi_gene()` — readable Gene fields

Compare each label with the preceding raw record: full name → `nomenclaturename`/`description`; chromosome → `chromosome`; map location → `maplocation`; aliases → `otheraliases`; summary → `summary`; MIM references → `mim`. Empty optional annotations should display `Unavailable`.


In [6]:
gene_fields = gr.get_ncbi_gene(symbol, organism)
pprint(gene_fields, sort_dicts=False)
assert gene_fields['Gene ID'] == gene_id


{'Gene ID': '348',
 'Symbol': 'APOE',
 'Organism': 'Homo sapiens',
 'Full name': 'apolipoprotein E',
 'Chromosome': '19',
 'Map location': '19q13.32',
 'Aliases': 'AD2, APO-E, ApoE4, LDLCQ5, LPG',
 'Summary': 'The protein encoded by this gene is a major apoprotein of the '
            'chylomicron. It binds to a specific liver and peripheral cell '
            'receptor, and is essential for the normal catabolism of '
            'triglyceride-rich lipoprotein constituents. This gene maps to '
            'chromosome 19 in a cluster with the related apolipoprotein C1 and '
            'C2 genes. Mutations in this gene result in familial '
            'dysbetalipoproteinemia, or type III hyperlipoproteinemia (HLP '
            'III), in which increased plasma cholesterol and triglycerides are '
            'the consequence of impaired clearance of chylomicron and VLDL '
            'remnants. [provided by RefSeq, Jun 2016]',
 'MIM references': '107741'}


## 5. `find_refseq_rna_ids()` — linked RNA identifiers

First inspect the real ELink response, then compare the function's ID list with `linksets → linksetdbs → links`. These numeric IDs are not the NM_/XM_ accession/version strings. The link group can include noncoding RNAs, which later mRNA parsing excludes.


In [7]:
link_parameters = {'dbfrom': 'gene', 'db': 'nuccore', 'id': gene_id,
                   'linkname': 'gene_nuccore_refseqrna', 'retmode': 'json'}
link_json = gr.get_ncbi_json('elink.fcgi', link_parameters)
print('RAW ELINK:')
pprint(link_json, sort_dicts=False)
rna_ids = gr.find_refseq_rna_ids(gene_id)
print('\nFUNCTION OUTPUT:')
pprint(rna_ids)


RAW ELINK:
{'header': {'type': 'elink', 'version': '0.3'},
 'linksets': [{'dbfrom': 'gene',
               'ids': ['348'],
               'linksetdbs': [{'dbto': 'nuccore',
                               'linkname': 'gene_nuccore_refseqrna',
                               'links': ['1889569396',
                                         '1676440320',
                                         '1676318530',
                                         '1676318097',
                                         '1519315198']}]}]}

FUNCTION OUTPUT:
['1889569396', '1676440320', '1676318530', '1676318097', '1519315198']


### Raw GenBank input for the next two parsers

This extra inspection cell fetches the actual linked records using `requests`, not Biopython. Change `record_index` to inspect another record and rerun the next two parser cells. Record order need not be accession order. The printed annotation omits the nucleotide sequence after ORIGIN, but the parsers receive the complete record.


In [8]:
assert rna_ids, 'No linked RNA records were returned; there is no GenBank record to inspect.'
time.sleep(0.34)
response = requests.get(gr.EUTILS_URL + 'efetch.fcgi', params={
    'db': 'nuccore', 'id': ','.join(rna_ids), 'rettype': 'gb', 'retmode': 'text',
}, timeout=30)
response.raise_for_status()
print('Raw API URL:', response.url)
raw_records = []
for record in re.split(r'(?m)^//\s*$', response.text):
    if record.strip():
        raw_records.append(record)
record_index = 0
selected_record = raw_records[record_index]
print('Fetched records:', len(raw_records), 'Selected index:', record_index)
print(selected_record.split('ORIGIN')[0])


Raw API URL: https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=nuccore&id=1889569396%2C1676440320%2C1676318530%2C1676318097%2C1519315198&rettype=gb&retmode=text
Fetched records: 5 Selected index: 0
LOCUS       NM_001302690            1244 bp    mRNA    linear   PRI 10-SEP-2026
DEFINITION  Homo sapiens apolipoprotein E (APOE), transcript variant 4, mRNA.
ACCESSION   NM_001302690
VERSION     NM_001302690.2
KEYWORDS    RefSeq.
SOURCE      Homo sapiens (human)
  ORGANISM  Homo sapiens
            Eukaryota; Metazoa; Chordata; Craniata; Vertebrata; Euteleostomi;
            Mammalia; Eutheria; Euarchontoglires; Primates; Haplorrhini;
            Catarrhini; Hominidae; Homo.
REFERENCE   1  (bases 1 to 1244)
  AUTHORS   Chen,L., He,X., Mao,L. and Liu,P.
  TITLE     APOE contributes to longitudinal impulse control disorders
            progression in Parkinson's disease
  JOURNAL   BMC Psychiatry 24 (1), 632 (2024)
   PUBMED   39334114
  REMARK    GeneRIF: APOE contributes to longit

## 6. `parse_cds_features()` — CDS annotations only

Compare every returned location, `protein_id` and `transl_table` with its CDS block in the raw annotation. Locations are 1-based inclusive positions on the transcript. Keep join/complement/partial notation. A missing `transl_table` is `Unavailable`, not a value inferred from the organism.


In [9]:
cds_fields = gr.parse_cds_features(selected_record)
pprint(cds_fields, sort_dicts=False)


[{'Location': '148..1101',
  'protein_id': 'NP_001289619.1',
  'transl_table': 'Unavailable'}]


## 7. `parse_refseq_mrna()` — one transcript

Compare VERSION, LOCUS length/date and the CDS blocks with the result. A noncoding record returns `None`. For the previously checked `NM_000041.4`, the values were 1166 nt, CDS 70..1023 and protein NP_000032.1. If this is another record, compare that record's own values instead.


In [10]:
selected_transcript = gr.parse_refseq_mrna(selected_record, gene_id, organism)
pprint(selected_transcript, sort_dicts=False)
if selected_transcript is not None:
    assert selected_transcript['CDS'] == cds_fields
    print('Online GenBank record:', 'https://www.ncbi.nlm.nih.gov/nuccore/'
          + selected_transcript['Accession/version'] + '?report=genbank')


{'Accession/version': 'NM_001302690.2',
 'Length (nt)': 1244,
 'Last modified': '10-SEP-2026',
 'CDS': [{'Location': '148..1101',
          'protein_id': 'NP_001289619.1',
          'transl_table': 'Unavailable'}]}
Online GenBank record: https://www.ncbi.nlm.nih.gov/nuccore/NM_001302690.2?report=genbank


## 8. `get_ncbi_nucleotide()` — all retained mRNAs

Inspect every returned transcript and its printed record link. Compare against all fetched GenBank records, including dates and every CDS; do not compare chromosome coordinates with these transcript-relative coordinates. A change in the live database between requests can require reloading both inputs.


In [11]:
transcript_fields = gr.get_ncbi_nucleotide(gene_id, organism)
pprint(transcript_fields, sort_dicts=False)
for transcript in transcript_fields:
    print('https://www.ncbi.nlm.nih.gov/nuccore/' + transcript['Accession/version'] + '?report=genbank')
print('Retained mRNAs:', len(transcript_fields))


[{'Accession/version': 'NM_000041.4',
  'Length (nt)': 1166,
  'Last modified': '09-SEP-2026',
  'CDS': [{'Location': '70..1023',
           'protein_id': 'NP_000032.1',
           'transl_table': 'Unavailable'}]},
 {'Accession/version': 'NM_001302688.2',
  'Length (nt)': 1248,
  'Last modified': '09-SEP-2026',
  'CDS': [{'Location': '74..1105',
           'protein_id': 'NP_001289617.1',
           'transl_table': 'Unavailable'}]},
 {'Accession/version': 'NM_001302689.2',
  'Length (nt)': 1144,
  'Last modified': '09-SEP-2026',
  'CDS': [{'Location': '48..1001',
           'protein_id': 'NP_001289618.1',
           'transl_table': 'Unavailable'}]},
 {'Accession/version': 'NM_001302690.2',
  'Length (nt)': 1244,
  'Last modified': '10-SEP-2026',
  'CDS': [{'Location': '148..1101',
           'protein_id': 'NP_001289619.1',
           'transl_table': 'Unavailable'}]},
 {'Accession/version': 'NM_001302691.2',
  'Length (nt)': 1181,
  'Last modified': '09-SEP-2026',
  'CDS': [{'Location': 

## 9. `search_uniprot_entries()` — raw search and exact matches

Fetch raw reviewed results separately, including every page, and then call the script's search function. If no exact reviewed match is found, this cell shows the unreviewed search instead, matching the approved fallback rule. Compare gene name, scientific species, accession and entry type. The browser API URLs request the same fields; the next cell gives an entry page link.


In [13]:
reviewed_entries = gr.search_uniprot_entries(symbol, organism, reviewed=True)
print('Reviewed function output:')
pprint(reviewed_entries, sort_dicts=False)
selected_reviewed = bool(reviewed_entries)
if selected_reviewed:
    selected_entries = reviewed_entries
else:
    selected_entries = gr.search_uniprot_entries(symbol, organism, reviewed=False)
    print('Unreviewed fallback function output:')
    pprint(selected_entries, sort_dicts=False)

status = 'true' if selected_reviewed else 'false'
parameters = {
    'query': f'gene_exact:"{symbol}" AND organism_name:"{organism}" AND reviewed:{status}',
    'format': 'json', 'size': 500,
    'fields': 'accession,reviewed,protein_name,gene_names,organism_name,organism_id,'
              'cc_function,length,ft_binding,cc_cofactor,cc_disease,xref_pdb,keyword',
}
url = gr.UNIPROT_URL
raw_uniprot_entries = []
while url:
    response = requests.get(url, params=parameters, timeout=30)
    response.raise_for_status()
    page = response.json()
    print('\nRAW API PAGE:', response.url)
    pprint(page, sort_dicts=False)
    raw_uniprot_entries.extend(page['results'])
    url = response.links.get('next', {}).get('url')
    parameters = None
for entry in selected_entries:
    assert entry in raw_uniprot_entries, 'Reload both responses if the database changed.'


Reviewed function output:
[{'entryType': 'UniProtKB reviewed (Swiss-Prot)',
  'primaryAccession': 'P02649',
  'organism': {'scientificName': 'Homo sapiens',
               'commonName': 'Human',
               'taxonId': 9606,
               'lineage': ['Eukaryota',
                           'Metazoa',
                           'Chordata',
                           'Craniata',
                           'Vertebrata',
                           'Euteleostomi',
                           'Mammalia',
                           'Eutheria',
                           'Euarchontoglires',
                           'Primates',
                           'Haplorrhini',
                           'Catarrhini',
                           'Hominidae',
                           'Homo']},
  'proteinDescription': {'recommendedName': {'fullName': {'evidences': [{'evidenceCode': 'ECO:0000305'}],
                                                          'value': 'Apolipoprotein '
                  

### Select the protein to inspect

Change `protein_index` to inspect each returned protein and rerun cells 10–12. Complete raw comments, binding features, PDB references and keywords were printed above; no annotation is inferred from the web page's shortened summary.


In [14]:
assert selected_entries, 'No matching UniProt entry was returned; there is no protein to inspect.'
protein_index = 0
selected_entry = selected_entries[protein_index]
print('Protein:', selected_entry['primaryAccession'])
print('Online entry:', 'https://www.uniprot.org/uniprotkb/' + selected_entry['primaryAccession'] + '/entry')


Protein: P02649
Online entry: https://www.uniprot.org/uniprotkb/P02649/entry


## 10. `uniprot_texts()` — annotation text values

Print the input text objects for each function comment and then the helper output. Values should be copied without rewriting the biology; evidence objects are not part of the returned strings. If there are no function comments, a separate labelled empty-input check still shows the helper's result.


In [15]:
function_comment_count = 0
for comment in selected_entry.get('comments', []):
    if comment.get('commentType') == 'FUNCTION':
        function_comment_count += 1
        text_objects = comment.get('texts', [])
        print('RAW TEXT OBJECTS:')
        pprint(text_objects, sort_dicts=False)
        print('FUNCTION OUTPUT:')
        pprint(gr.uniprot_texts(text_objects))
if function_comment_count == 0:
    print('No live function comment. Empty-input result:', gr.uniprot_texts([]))


RAW TEXT OBJECTS:
[{'evidences': [{'evidenceCode': 'ECO:0000250',
                 'source': 'UniProtKB',
                 'id': 'P08226'},
                {'evidenceCode': 'ECO:0000269',
                 'source': 'PubMed',
                 'id': '12950167'},
                {'evidenceCode': 'ECO:0000269',
                 'source': 'PubMed',
                 'id': '14754908'},
                {'evidenceCode': 'ECO:0000269',
                 'source': 'PubMed',
                 'id': '1530612'},
                {'evidenceCode': 'ECO:0000269',
                 'source': 'PubMed',
                 'id': '1911868'},
                {'evidenceCode': 'ECO:0000269',
                 'source': 'PubMed',
                 'id': '1917954'},
                {'evidenceCode': 'ECO:0000269',
                 'source': 'PubMed',
                 'id': '20030366'},
                {'evidenceCode': 'ECO:0000269',
                 'source': 'PubMed',
                 'id': '20303980'},
                

## 11. `parse_uniprot_comments()` — functions, cofactors and diseases

Compare with raw `comments` above. Inspect every function paragraph, cofactor name/note and disease description/note. In particular, susceptibility must not be rewritten as definite causation. Missing cofactor annotations do not establish biological absence.


In [16]:
protein_comments = gr.parse_uniprot_comments(selected_entry)
pprint(protein_comments, sort_dicts=False)


{'Function': ['APOE is an apolipoprotein, a protein associating with lipid '
              'particles, that mainly functions in lipoprotein-mediated lipid '
              'transport between organs via the plasma and interstitial fluids '
              '(PubMed:14754908, PubMed:1911868, PubMed:6860692). APOE is a '
              'core component of plasma lipoproteins and is involved in their '
              'production, conversion and clearance (PubMed:14754908, '
              'PubMed:1911868, PubMed:1917954, PubMed:23620513, '
              'PubMed:2762297, PubMed:6860692, PubMed:9395455). '
              'Apolipoproteins are amphipathic molecules that interact both '
              'with lipids of the lipoprotein particle core and the aqueous '
              'environment of the plasma (PubMed:2762297, PubMed:6860692, '
              'PubMed:9395455). As such, APOE associates with chylomicrons, '
              'chylomicron remnants, very low density lipoproteins (VLDL) and '
          

## 12. `parse_uniprot_entry()` — one readable protein entry

Compare all ten fields with the raw JSON: accession/type, protein name, sequence length, comments, binding-site ligands, PDB IDs and keyword names. The previous P02649 example had 317 aa, heparin, 29 PDB IDs and 30 keywords; check current data rather than fixing those counts as permanent truths.


In [17]:
selected_protein_fields = gr.parse_uniprot_entry(selected_entry)
pprint(selected_protein_fields, sort_dicts=False)
print('Raw binding-site annotations:')
pprint(selected_entry.get('features', []), sort_dicts=False)
print('Raw PDB/keyword counts:', len(selected_entry.get('uniProtKBCrossReferences', [])),
      len(selected_entry.get('keywords', [])))


{'Accession': 'P02649',
 'Review status': 'UniProtKB reviewed (Swiss-Prot)',
 'Protein name': 'Apolipoprotein E',
 'Function': 'APOE is an apolipoprotein, a protein associating with lipid '
             'particles, that mainly functions in lipoprotein-mediated lipid '
             'transport between organs via the plasma and interstitial fluids '
             '(PubMed:14754908, PubMed:1911868, PubMed:6860692). APOE is a '
             'core component of plasma lipoproteins and is involved in their '
             'production, conversion and clearance (PubMed:14754908, '
             'PubMed:1911868, PubMed:1917954, PubMed:23620513, PubMed:2762297, '
             'PubMed:6860692, PubMed:9395455). Apolipoproteins are amphipathic '
             'molecules that interact both with lipids of the lipoprotein '
             'particle core and the aqueous environment of the plasma '
             '(PubMed:2762297, PubMed:6860692, PubMed:9395455). As such, APOE '
             'associates with chyl

## 13. `get_uniprot_proteins()` — selection and all readable proteins

Inspect all entries and their review labels. When reviewed matches exist, unreviewed matches are excluded. Otherwise all exact unreviewed matches are retained. Results are accession-sorted. Empty output means no exact match was returned, not that the organism has no proteins.


In [18]:
protein_fields = gr.get_uniprot_proteins(symbol, organism)
pprint(protein_fields, sort_dicts=False)
for protein in protein_fields:
    print('https://www.uniprot.org/uniprotkb/' + protein['Accession'] + '/entry')


[{'Accession': 'P02649',
  'Review status': 'UniProtKB reviewed (Swiss-Prot)',
  'Protein name': 'Apolipoprotein E',
  'Function': 'APOE is an apolipoprotein, a protein associating with lipid '
              'particles, that mainly functions in lipoprotein-mediated lipid '
              'transport between organs via the plasma and interstitial fluids '
              '(PubMed:14754908, PubMed:1911868, PubMed:6860692). APOE is a '
              'core component of plasma lipoproteins and is involved in their '
              'production, conversion and clearance (PubMed:14754908, '
              'PubMed:1911868, PubMed:1917954, PubMed:23620513, '
              'PubMed:2762297, PubMed:6860692, PubMed:9395455). '
              'Apolipoproteins are amphipathic molecules that interact both '
              'with lipids of the lipoprotein particle core and the aqueous '
              'environment of the plasma (PubMed:2762297, PubMed:6860692, '
              'PubMed:9395455). As such, APOE assoc

## 14. `fetch_ensembl_gene()` — expanded raw gene

This function returns the validated raw response. Print the gene fields and transcript ID/version/canonical flags while omitting bulky exon and translation objects from the display; the complete response remains in `ensembl_gene`. Compare the online gene page on the **same assembly**. This request has a 90-second read timeout because Ensembl previously responded slowly.


In [19]:
ensembl_gene = gr.fetch_ensembl_gene(symbol, organism)
print('GENE FIELDS:')
for field, value in ensembl_gene.items():
    if field != 'Transcript':
        print(field, ':', value)
print('\nTRANSCRIPTS:')
for transcript in ensembl_gene.get('Transcript', []):
    print(transcript.get('id'), transcript.get('version'), transcript.get('is_canonical'))
print('Online gene:', 'https://www.ensembl.org/' + species + '/Gene/Summary?g=' + ensembl_gene['id'])
print('Raw API:', requests.Request('GET', gr.ENSEMBL_URL + species + '/' + symbol,
      params={'expand': 1, 'content-type': 'application/json'}).prepare().url)


GENE FIELDS:
id : ENSG00000130203
source : ensembl_havana
object_type : Gene
start : 44903787
strand : 1
biotype : protein_coding
logic_name : ensembl_havana_gene_homo_sapiens
seq_region_name : 19
description : apolipoprotein E [Source:HGNC Symbol;Acc:HGNC:613]
species : homo_sapiens
display_name : APOE
assembly_name : GRCh38
version : 12
canonical_transcript : ENST00000252486.9
end : 44909396
db_type : core

TRANSCRIPTS:
ENST00000446996 5 0
ENST00000485628 2 0
ENST00000252486 9 1
ENST00000434152 5 0
ENST00000425718 1 0
ENST00000864817 2 0
ENST00000864820 2 0
ENST00000864822 2 0
ENST00000864824 2 0
ENST00000864826 2 0
ENST00000864828 2 0
ENST00000864829 2 0
ENST00000864830 2 0
ENST00000864831 2 0
ENST00000864832 2 0
ENST00000864833 2 0
ENST00000864834 2 0
ENST00000864835 2 0
ENST00000864836 2 0
ENST00000864837 1 0
ENST00000864838 2 0
ENST00000864839 2 0
ENST00000864840 2 0
ENST00000864841 2 0
ENST00000864842 2 0
ENST00000864843 2 0
ENST00000864844 2 0
ENST00000864845 2 0
ENST0000086484

## 15. `parse_ensembl_gene()` — readable coordinates and transcript count

Compare each field with the raw data, counting the expanded transcript list. Missing lists are `Unavailable`, explicitly empty lists are zero. Keep the canonical transcript version. Coordinates are 1-based inclusive genomic positions; strand -1 does not reverse the numeric start/end order. RefSeq mRNA and Ensembl transcript sets need not have equal counts.


In [20]:
ensembl_fields = gr.parse_ensembl_gene(ensembl_gene)
pprint(ensembl_fields, sort_dicts=False)
if isinstance(ensembl_gene.get('Transcript'), list):
    print('Direct raw-list count:', len(ensembl_gene['Transcript']))
    assert ensembl_fields['Transcript count'] == len(ensembl_gene['Transcript'])


{'Gene ID': 'ENSG00000130203',
 'Assembly': 'GRCh38',
 'Sequence region': '19',
 'Start (1-based, inclusive)': 44903787,
 'End (1-based, inclusive)': 44909396,
 'Strand (+1 forward, -1 reverse)': 1,
 'Transcript count': 39,
 'Canonical transcript': 'ENST00000252486.9',
 'Biotype': 'protein_coding'}
Direct raw-list count: 39


## 16. `markdown_text()` — literal punctuation

This is a formatting check, not an online lookup. Expected raw Markdown: a backslash before the allele asterisk and accession underscore. A Markdown viewer displays the original characters rather than interpreting them as emphasis.


In [21]:
annotation_example = 'APOE*4 / NP_000032.1'
escaped_example = gr.markdown_text(annotation_example)
print('Input:', annotation_example)
print('Function output:', escaped_example)
assert escaped_example == 'APOE\\*4 / NP\\_000032.1'


Input: APOE*4 / NP_000032.1
Function output: APOE\*4 / NP\_000032.1


## 17. `format_markdown_fields()` — labels and paragraphs

Input is the already inspected protein. Compare the Markdown source below with the dictionary from cell 12: labels and blank lines are formatting, while the annotation content should remain intact. Missing values must remain visible.


In [22]:
formatted_protein = gr.format_markdown_fields(selected_protein_fields)
print(formatted_protein)


**Accession:** P02649

**Review status:** UniProtKB reviewed (Swiss-Prot)

**Protein name:** Apolipoprotein E

**Function:** APOE is an apolipoprotein, a protein associating with lipid particles, that mainly functions in lipoprotein-mediated lipid transport between organs via the plasma and interstitial fluids (PubMed:14754908, PubMed:1911868, PubMed:6860692). APOE is a core component of plasma lipoproteins and is involved in their production, conversion and clearance (PubMed:14754908, PubMed:1911868, PubMed:1917954, PubMed:23620513, PubMed:2762297, PubMed:6860692, PubMed:9395455). Apolipoproteins are amphipathic molecules that interact both with lipids of the lipoprotein particle core and the aqueous environment of the plasma (PubMed:2762297, PubMed:6860692, PubMed:9395455). As such, APOE associates with chylomicrons, chylomicron remnants, very low density lipoproteins (VLDL) and intermediate density lipoproteins (IDL) but shows a preferential binding to high-density lipoproteins (HDL

## 18. `build_report()` — complete report in memory

Inspect all four sections, all transcript/CDS blocks and all selected proteins. Compare with the preceding function outputs and online links. This function does not write a file. The optional diff is against your earlier saved report; differences can reflect database updates and require review rather than automatically proving a bug.


In [23]:
report = gr.build_report(symbol, organism, gene_fields, transcript_fields, protein_fields, ensembl_fields)
print(report)
for section in ('## NCBI Gene', '## NCBI Nucleotide', '## UniProtKB', '## Ensembl'):
    assert section in report
if saved_report_path.is_file():
    saved_report = saved_report_path.read_text(encoding='utf-8')
    if saved_report == report:
        print('Matches the saved report exactly.')
    else:
        print('DIFFERENCES FROM THE SAVED REPORT — review against current online records:')
        differences = difflib.unified_diff(saved_report.splitlines(), report.splitlines(),
                                          fromfile='saved report', tofile='current live report', lineterm='')
        for line in differences:
            print(line)


# Gene report: APOE

**Organism:** Homo sapiens

Unavailable means the requested annotation was not returned; it does not prove biological absence.

## NCBI Gene

**Gene ID:** 348

**Symbol:** APOE

**Organism:** Homo sapiens

**Full name:** apolipoprotein E

**Chromosome:** 19

**Map location:** 19q13.32

**Aliases:** AD2, APO-E, ApoE4, LDLCQ5, LPG

**Summary:** The protein encoded by this gene is a major apoprotein of the chylomicron. It binds to a specific liver and peripheral cell receptor, and is essential for the normal catabolism of triglyceride-rich lipoprotein constituents. This gene maps to chromosome 19 in a cluster with the related apolipoprotein C1 and C2 genes. Mutations in this gene result in familial dysbetalipoproteinemia, or type III hyperlipoproteinemia (HLP III), in which increased plasma cholesterol and triglycerides are the consequence of impaired clearance of chylomicron and VLDL remnants. \[provided by RefSeq, Jun 2016\]

**MIM references:** 107741

## NCBI Nucl

## 19. `write_report()` — temporary output and rerun

Expected: a gene/organism Markdown filename and exact UTF-8 report contents. The second write replaces the same file. Both writes occur in a temporary directory and do not touch `assignment_2/reports/`.


In [24]:
with tempfile.TemporaryDirectory() as temporary:
    temporary_out = Path(temporary) / 'reports'
    written_path = gr.write_report(temporary_out, symbol, organism, report)
    print('Function output:', written_path)
    assert written_path.read_text(encoding='utf-8') == report
    rewritten_path = gr.write_report(temporary_out, symbol, organism, report)
    assert rewritten_path == written_path
    assert rewritten_path.read_text(encoding='utf-8') == report
    print('File contents match the inspected report; replacement check passed.')


Function output: C:\Users\misie\AppData\Local\Temp\tmp1oxpz9lb\reports\APOE_Homo_sapiens.md
File contents match the inspected report; replacement check passed.


## 20. `main()` — actual command-line entry point

This subprocess invokes `main()` through the real script, with the same Python as the notebook. It performs **fresh online lookups**, so it may take longer and can differ if annotations change. Inspect exit status and output; success must produce exactly one Markdown report with four database sections. The output directory is temporary.


In [25]:
with tempfile.TemporaryDirectory() as temporary:
    command = [sys.executable, '-B', 'genereport.py', symbol, '--organism', organism, '--out', temporary]
    result = subprocess.run(command, cwd=assignment_dir, capture_output=True, text=True)
    print('Exit status:', result.returncode)
    print('Standard output:', result.stdout)
    print('Standard error:', result.stderr)
    assert result.returncode == 0, 'The live CLI failed; inspect its error before continuing.'
    report_paths = list(Path(temporary).glob('*.md'))
    assert len(report_paths) == 1
    cli_report = report_paths[0].read_text(encoding='utf-8')
    print(cli_report)
    for section in ('## NCBI Gene', '## NCBI Nucleotide', '## UniProtKB', '## Ensembl'):
        assert section in cli_report
    print('Matches the earlier in-memory live report:', cli_report == report)


Exit status: 0
Standard output: Wrote report to C:\Users\misie\AppData\Local\Temp\tmp02ovpaqp\APOE_Homo_sapiens.md.

Standard error: 
# Gene report: APOE

**Organism:** Homo sapiens

Unavailable means the requested annotation was not returned; it does not prove biological absence.

## NCBI Gene

**Gene ID:** 348

**Symbol:** APOE

**Organism:** Homo sapiens

**Full name:** apolipoprotein E

**Chromosome:** 19

**Map location:** 19q13.32

**Aliases:** AD2, APO-E, ApoE4, LDLCQ5, LPG

**Summary:** The protein encoded by this gene is a major apoprotein of the chylomicron. It binds to a specific liver and peripheral cell receptor, and is essential for the normal catabolism of triglyceride-rich lipoprotein constituents. This gene maps to chromosome 19 in a cluster with the related apolipoprotein C1 and C2 genes. Mutations in this gene result in familial dysbetalipoproteinemia, or type III hyperlipoproteinemia (HLP III), in which increased plasma cholesterol and triglycerides are the conseque

## Final check and interpretation

Expected: your script and pre-existing saved report have identical hashes. Raw-to-parsed equality checks establish extraction consistency; they are not independent proof that an online database is biologically complete or that every annotation is correct. Online pages may show fewer details or a newer annotation release, so compare accession/version and assembly before interpreting a difference. Restart the kernel after editing the script.


In [26]:
for path, original_hash in protected_hashes.items():
    current_hash = hashlib.sha256(path.read_bytes()).hexdigest()
    assert current_hash == original_hash, str(path) + ' changed during the notebook.'
print('The original script and saved report are unchanged.')


The original script and saved report are unchanged.
